In [ ]:
import os
import time
import warnings
from datetime import datetime, timedelta, timezone
import io
import csv
import pandas as pd
import requests
import xml.etree.ElementTree as ET
from google.colab import drive
from IPython.display import display

warnings.filterwarnings('ignore')
drive.mount('/content/drive', force_remount=True)

from getpass import getpass
API_AUTH_KEY = os.environ.get("WORK24_API_KEY") or getpass("고용24 API 인증키: ")

# KST 시간대 설정
def get_kst_now():
    try:
        from zoneinfo import ZoneInfo
        return datetime.now(ZoneInfo("Asia/Seoul"))
    except ImportError:
        try:
            import pytz
            kst = pytz.timezone('Asia/Seoul')
            return datetime.now(kst)
        except ImportError:
            kst = timezone(timedelta(hours=9))
            return datetime.now(kst)

# --- 데이터 수집 클래스 정의 ---
class Work24DetailCollector:
    """채용 상세정보 API 수집기"""
    def __init__(self, auth_key: str):
        self.auth_key = auth_key
        self.base_url = "https://www.work24.go.kr/cm/openApi/call/wk/callOpenApiSvcInfo210D01.do"
        self.session = requests.Session()

    def parse_detail_xml(self, xml_content: str) -> dict:
        try:
            root = ET.fromstring(xml_content)
            if root.find('.//wantedAuthNo') is None:
                error_message = root.find('.//message').text if root.find('.//message') is not None else "알 수 없는 오류"
                print(f"API 오류 응답: {error_message}")
                return {}
            detail_data = {}
            for elem in root.iter():
                if elem.tag == 'keywordList':
                    keywords = [k.text for k in elem.findall('.//srchKeywordNm') if k.text]
                    detail_data['keywords'] = ', '.join(keywords)
                elif elem.tag == 'corpAttachList':
                    attach_files = [f.text for f in elem.findall('.//attachFileUrl') if f.text]
                    detail_data['corpAttachFiles'] = ', '.join(attach_files)
                elif elem.text and elem.text.strip():
                    detail_data[elem.tag] = elem.text.strip()
            return detail_data
        except ET.ParseError as e:
            print(f"XML 파싱 오류: {e}")
            return {}

    def fetch_single_detail(self, wanted_auth_no: str) -> dict:
        params = {
            'authKey': self.auth_key, 'callTp': 'D', 'returnType': 'XML',
            'wantedAuthNo': wanted_auth_no, 'infoSvc': 'VALIDATION'
        }
        try:
            response = self.session.get(self.base_url, params=params, timeout=30)
            response.raise_for_status()
            return self.parse_detail_xml(response.text)
        except requests.exceptions.RequestException as e:
            print(f"API 요청 오류: {e}")
            return {}

# --- 메인 실행 함수 정의 ---
def run_and_save_details():
    """worknet_job.csv를 읽어 등록일(regDt)별로 상세정보를 중복 체크 후 수집/저장하는 메인 함수"""
    detail_collector = Work24DetailCollector(API_AUTH_KEY)

    save_path = '/content/drive/My Drive/Colab Notebooks/work24/'

    # KST 시간 기준으로 오늘 날짜 파일명 자동 생성
    kst_now = get_kst_now()
    today_kst = kst_now.strftime('%Y%m%d')
    master_file_path = os.path.join(save_path, f'worknet_jobs_{today_kst}.csv')

    print(f"채용 상세정보 수집을 시작... (KST: {kst_now.strftime('%Y-%m-%d %H:%M:%S')})")
    print(f"대상 파일: worknet_jobs_{today_kst}.csv")

    if not os.path.exists(master_file_path):
        print(f"worknet_job 파일을 찾을 수 없음.: {master_file_path}")

        # 사용 가능한 파일 목록 표시
        print("\n사용 가능한 파일들:")
        try:
            for file in os.listdir(save_path):
                if file.startswith('worknet_jobs_') and file.endswith('.csv') and not 'details' in file:
                    print(f"  - {file}")
        except:
            pass

        # 수동으로 파일명을 지정할 수 있도록 안내
        print(f"\n다른 날짜의 파일을 사용하려면 master_file_path를 직접 수정하세요.")
        print(f"현재 KST 기준 오늘: {today_kst}")
        return

    try:
        cleaned_data = []
        with open(master_file_path, 'r', encoding='utf-8-sig', newline='') as f:
            reader = csv.reader(f)
            header = next(reader)
            num_header_cols = len(header)
            for row in reader:
                if len(row) >= num_header_cols:
                    cleaned_data.append(row[:num_header_cols])
        master_df = pd.DataFrame(cleaned_data, columns=header)
        print(f"worknet_job 파일 읽기 성공: 총 {len(master_df)}건의 데이터를 읽음.")
    except Exception as e:
        print(f"CSV 파일 처리 중 오류: {e}")
        return

    if 'regDt' not in master_df.columns or 'wantedAuthNo' not in master_df.columns:
        print("'regDt' 또는 'wantedAuthNo' 컬럼이 master 파일에 존재하지 않음.")
        return

    unique_dates = master_df['regDt'].dropna().unique()
    print(f"총 {len(unique_dates)}개의 날짜에 대해 수집을 진행: {sorted(unique_dates)}")

    for reg_date in sorted(unique_dates):
        print("-" * 40)
        print(f"날짜 [{reg_date}]의 상세정보 수집 시작...")

        # --- 중복 체크 로직 시작 ---
        safe_reg_date = str(reg_date).replace('-', '').replace(' ', '_').replace(':', '')
        detail_filename = f'worknet_jobs_details_{safe_reg_date}.csv'
        detail_file_path = os.path.join(save_path, detail_filename)

        existing_detail_ids = set()
        if os.path.exists(detail_file_path):
            try:
                print(f"기존 상세정보 파일 '{detail_filename}' 확인.")
                existing_details_df = pd.read_csv(detail_file_path)
                if 'wantedAuthNo' in existing_details_df.columns:
                    existing_detail_ids = set(existing_details_df['wantedAuthNo'].dropna().astype(str))
                print(f"{len(existing_detail_ids)}개의 기존 상세정보 확인.")
            except Exception as e:
                print(f"기존 상세정보 파일 읽기 오류: {e}. 새로 생성될 수 있음")

        daily_df = master_df[master_df['regDt'] == reg_date]
        auth_nos_from_master = daily_df['wantedAuthNo'].dropna().astype(str).tolist()

        # 이미 수집된 ID를 제외한 새로운 ID 목록 생성
        auth_nos_to_fetch = [auth_no for auth_no in auth_nos_from_master if auth_no not in existing_detail_ids]

        if not auth_nos_to_fetch:
            print(f"모든 공고의 상세정보 이미 수집완료. 다음 날짜로 패스")
            continue

        print(f"총 {len(auth_nos_from_master)}개 공고 중 {len(auth_nos_to_fetch)}개의 신규 상세정보 수집.")

        newly_collected_details = []
        start_time = get_kst_now()
        print(f"수집 시작 시간 (KST): {start_time.strftime('%H:%M:%S')}")

        for i, auth_no in enumerate(auth_nos_to_fetch):
            print(f"   > [{i+1}/{len(auth_nos_to_fetch)}] 구인인증번호 '{auth_no}' 수집 중...")
            detail_data = detail_collector.fetch_single_detail(auth_no)
            if detail_data:
                newly_collected_details.append(detail_data)
            time.sleep(0.5)

        if not newly_collected_details:
            print("새로 수집된 상세정보 없음.")
            continue

        new_details_df = pd.DataFrame(newly_collected_details)

        # KST 시간으로 저장 완료 시간 표시
        save_time = get_kst_now()

        if os.path.exists(detail_file_path):
            new_details_df.to_csv(detail_file_path, mode='a', header=False, index=False, encoding='utf-8-sig')
            print(f"날짜 [{reg_date}]의 신규 상세정보 {len(new_details_df)}건을 기존 파일에 추가!")
        else:
            new_details_df.to_csv(detail_file_path, index=False, encoding='utf-8-sig')
            print(f"날짜 [{reg_date}]의 상세정보 {len(new_details_df)}건을 새 파일로 저장!")

        print(f"저장 경로: {detail_file_path}")
        print(f"저장 완료 시간 (KST): {save_time.strftime('%Y-%m-%d %H:%M:%S')}")

        # 소요 시간 계산
        elapsed_time = (save_time - start_time).total_seconds()
        print(f"이 날짜 처리 소요 시간: {elapsed_time:.1f}초")

        print(f"\n[{reg_date}] 새로 추가된 데이터 미리보기")
        display(new_details_df.head())

    print("-" * 40)
    final_time = get_kst_now()
    print(f"모든 날짜에 대한 상세정보 수집 완료. (KST: {final_time.strftime('%Y-%m-%d %H:%M:%S')})")


# --- 스크립트 실행 ---
print("=== 고용24 상세정보 수집기 ===")
run_and_save_details()